In [ ]:
###################################################################################

###Code for: "A semi-automated method for data extraction and high-density 2D geometric morphometrics of archaeological and heritage artefacts"
###Written by: Emily C. Watt
###Date last modified: July 2026

###################################################################################

###Written in Python version: 3.13.13

###Written using the following package versions:
#matplotlib==3.10.1
#numpy==2.2.5
#opencv-python==4.13.0.92  (cv2) 
#pandas==3.0.0
#pillow==11.2.1  (PIL)
#svgwrite==1.4.3
#tqdm==4.67.1
#pytesseract==0.3.13

###################################################################################

#Note that in an effort to ensure this script is useable by those with little or no prior coding knowledge, this code is heavily commented with prompts to edit where required

#The lines of code marked "###CHANGE HERE" may require editing by the user, and include: filepaths to input and output files/folders,
#                                                                                        toggling options on and off (True/False), 
#                                                                                        altering values according to own dataset

#Lines beginning with a # are comments and will not be executed when the code cell is run. This can be used to stop the execution of lines that you don't want to run (e.g., package installation after you have already done it)

###################################################################################

###In this script:
###     1. Standardise the DPI of the input images (if required) - can also done manually in image processing software, such as Inkscape
###     2. Extract single-object images from multi-object images (e.g., single letters from a page of text) - skip if your dataset is a single object per image
###     3. Optical Characters Recognition (OCR) sorting of individual character images into discrete folders - skip if your datasset is not text-based
###     4. Manual sorting using Graphic User Interface (GUI) - skip if no further sorting required, or your dataset is not text-based
###     5. Binarise the individual character images and fill internal holes within the contours
###     6. Transform the images (if required)
###     7. Extract the contours of the objects in the images and save the coordinates as a .csv file

###################################################################################

In [ ]:
#Install packages if you haven't already (uncomment the lines below to run by removing the # at the beginning of each line)
#The packages only need to be installed once. After installation, the lines can be commentted out with a # at the start, or the installationlines can be removed fully
#They require pip, which is the package installer for Python. If pip is not installed, you can install it by following the instructions here: https://pip.pypa.io/en/stable/installation/
#To run a cell in a Jupyter notebook (.ipynb), click on the cell and press Ctrl + ALt + Enter, or Shift + Enter, or click the "Run" button at the top right of the cell of code you want to run
#If you are running this code as a Python script (.py), you can run the installation commands in your terminal or command prompt

#Note: If running this code in a Jupyter notebook, you should restart the kernel after installing packages to ensure they are fully installed

#!pip install matplotlib
#!pip install numpy
#!pip install opencv-python
#!pip install pandas
#!pip install Pillow
#!pip install pytesseract
#!pip install svgwrite
#!pip install tqdm

#Important! Check that the correct version of opencv-python has installed
#pip list | findstr opencv-python
#If you see an entry for opencv-python-headless, it will not allow any GUI to run, so you will have to uninstall and reinstall opencv-python
#%pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python
#pip install opencv-python
#Check again, and this time the list should not include opencv-python-headless
#pip list | findstr opencv
#Finally, make sure to restart the kernell after installing packages (or fully close and reopen the Juypter notebook)

In [ ]:
###     1. Standardise the DPI of the input images (if required)
###         (can also done manually in image processing software, such as Inkscape)

#load packages
import os #for file handling
from PIL import Image #for image handling
from tqdm import tqdm #for progress bars
from pathlib import Path #for defining filepaths

#specify input and output folder locations
try:
    #for .py scripts, where the folder with the image data is also where the script is saved
    root_dir = Path(__file__).resolve().parent
except NameError:
    #for in Jupyter notebooks, where this calls on the current working directory (the folder from which this session was opened)
    root_dir = Path.cwd()

input_folder = root_dir / "clean_images"
output_folder = root_dir / "standardised_dpi" #this folder is created as part of this script, you do not have to manually create it

#you could also specify your folders as follows, if your code is not saved in the same location as your image data
#input_folder = r'C:\your\file\path\folder_with_images'                              ### CHANGE HERE
#output_folder = r'C:\your\file\path\standardised_dpi'                               ### CHANGE HERE

#create the output folder if it doesn't already exist
os.makedirs(output_folder, exist_ok=True)

#define default output DPI you want for the output images
DEFAULT_DPI = 500                                                                   ### CHANGE HERE

#create functions
#identify image DPI, uses DEFAULT_DPI if DPI metadata is not available in the image file
def get_image_dpi(image_path, default_dpi=DEFAULT_DPI):
    try:
        with Image.open(image_path) as img:
            dpi = img.info.get("dpi")
            if dpi and isinstance(dpi, (tuple, list)) and len(dpi) == 2:
                return dpi
    except Exception as e:
        print(f"Warning: Can't read DPI from {image_path}: {e}, so the default DPI will be used. You could try checking the DPI in the image properties using Inkscape or another image software.")
    return (default_dpi, default_dpi)

#resize the image to adjust for the target DPI while maintaining the physical size
def resize_image_resolution(image, target_dpi, original_dpi):
    #get the original image size in inches based on original DPI
    original_width, original_height = image.size
    original_width_inch = original_width / original_dpi[0]
    original_height_inch = original_height / original_dpi[1]
    
    #calculate the new pixel dimensions based on the target DPI
    new_width = int(original_width_inch * target_dpi)
    new_height = int(original_height_inch * target_dpi)
    
    #resize the image to the new dimensions
    resized_image = image.resize((new_width, new_height), Image.LANCZOS)
    return resized_image

#save the image with new DPI metadata
def save_with_dpi(image, save_path, dpi):
    image.save(save_path, dpi=dpi)

#process a single image - this is nested inside the process_folder function
def process_image(image_path, output_dir, target_dpi=DEFAULT_DPI):
    image_name = os.path.splitext(os.path.basename(image_path))[0]
    input_dpi = get_image_dpi(image_path, target_dpi)

    with Image.open(image_path) as img:
        #resize image if needed based on DPI
        resized_image = resize_image_resolution(img, target_dpi, input_dpi)

        #save with new DPI
        save_path = os.path.join(output_dir, f"{image_name}_{target_dpi}dpi{os.path.splitext(image_path)[1]}")
        save_with_dpi(resized_image, save_path, (target_dpi, target_dpi))

        print(f"Saved adjusted image: {save_path}")

#process a folder of images
def process_folder(input_folder, output_folder, target_dpi=DEFAULT_DPI):
    os.makedirs(output_folder, exist_ok=True)
    supported_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')

    images = [f for f in os.listdir(input_folder) if f.lower().endswith(supported_extensions)]
    total_images = len(images)

    if total_images == 0:
        print("No valid images found in the specified folder.")
        return

    for filename in tqdm(images, desc="Processing Images", unit="image"):
        image_path = os.path.join(input_folder, filename)
        process_image(image_path, output_folder, target_dpi)


#resize images in the input folder to the target DPI and save them in the output folder
process_folder(input_folder, output_folder, target_dpi=DEFAULT_DPI)

In [ ]:
###     2. Extract single-object images from multi-object images (e.g., single letters from a page of text)
###         (skip if your dataset is a single object per image)

#load packages
import os #for file handling
from PIL import Image #for image handling
import cv2 as cv #OpenCV for image processing
import numpy as np #NumPy for numerical operations
from tqdm import tqdm #for progress bars
import matplotlib.pyplot as plt #for plotting
from pathlib import Path #for defining filepaths

#specify input and output folder locations
try:
    #for .py scripts, where the folder with the image data is also where the script is saved
    root_dir = Path(__file__).resolve().parent
except NameError:
    #for in Jupyter notebooks, where this calls on the current working directory (the folder from which this session was opened)
    root_dir = Path.cwd()

input_folder = root_dir / "standardised_dpi"
output_folder = root_dir / "extracted" #this folder is created as part of this script, you do not have to manually create it

#you could also specify your folders as follows, if your code is not saved in the same location as your image data
#input_folder = r"C:\your\file\path\standardised_dpi"                                    ### CHANGE HERE
#output_folder = r"C:\your\file\path\extracted"                                          ### CHANGE HERE

#define default output DPI you want for the output images
DEFAULT_DPI = 500                                                                       ### CHANGE HERE

#create functions

#function to identify image DPI
#uses DEFAULT_DPI if DPI metadata is not available in the image file
def get_image_dpi(image_path, default_dpi=DEFAULT_DPI):
    try:
        with Image.open(image_path) as img:
            dpi = img.info.get("dpi")
            if dpi and isinstance(dpi, (tuple, list)) and len(dpi) == 2:
                return dpi
    except Exception as e:
        print(f"Warning: Can't read DPI from {image_path}: {e}, so the default DPI will be used. You could try checking the DPI in the image properties using Inkscape or another image software.")
    return (default_dpi, default_dpi)

#save the cropped image with DPI metadata
def save_with_dpi(crop, save_path, dpi):
    im = Image.fromarray(cv.cvtColor(crop, cv.COLOR_BGR2RGB))
    im.save(save_path, dpi=dpi)

def process_image(image_path, output_dir, default_dpi=DEFAULT_DPI):
    image_name = os.path.splitext(os.path.basename(image_path))[0]
    input_dpi = get_image_dpi(image_path, default_dpi)

    image = cv.imread(image_path)
    if image is None:
        print(f"Warning: Failed to read {image_path}, so this image has been skipped. Please check the input path and file format are as you expect!")
        return

    gray = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
    _, thresh = cv.threshold(gray, 128, 255, cv.THRESH_BINARY_INV)                      ### CHANGE HERE - depending on the colours of the onjects and background in your image, you may need to change cv.THRESH_BINARY_INV to cv.THRESH_BINARY, or adjust the threshold value (127) to better separate objects from the background
    contours, _ = cv.findContours(thresh, cv.RETR_TREE, cv.CHAIN_APPROX_SIMPLE) # cv.RETR_TREE finds all contours in an image and determines their internal hierarchy
                                                                                # cv.CHAIN_APPROX_SIMPLE only stores endpoints of contours rather than every point in the contour path, this speeds up runtime but could be changed to cv.CHAIN_APPROX_NONE to store all points

    approx_contours = []
    for contour in contours:
        epsilon = 0.005 * cv.arcLength(contour, True)
        approx = cv.approxPolyDP(contour, epsilon, True)
        approx_contours.append(approx)

    lines = []
    min_area = 100                                                                      ### CHANGE HERE - adjust to filter out small contours (e.g., background noise in input image)
    min_width = 10                                                                      ### CHANGE HERE - can also adjust minimum width of contours
    min_height = 10                                                                     ### CHANGE HERE - and minimum height of contours, if area is less useful to your data

    for contour in approx_contours:
        x, y, w, h = cv.boundingRect(contour)
        cy = y + h // 2
        placed = False

        if w > min_width and h > min_height and cv.contourArea(contour) > min_area:
            for line in lines:
                avg_y = np.mean([b[1] + b[3] // 2 for b in line])
                avg_height = np.mean([b[3] for b in line])
                if abs(cy - avg_y) < avg_height * 0.6:
                    line.append((x, y, w, h))
                    placed = True
                    break
            if not placed:
                lines.append([(x, y, w, h)])

    for line in lines:
        line.sort(key=lambda b: b[0])
    lines.sort(key=lambda line: line[0][1])

    for line_idx, line in enumerate(lines):
        for pos_idx, (x, y, w, h) in enumerate(line):
            margin = 10                                                                ### CHANGE HERE - push the bounding box outwards by a set margin for each contour, this helps to catch any small details that the bounding box inadvertently cuts off
                                                                                                        #a smaller margin (5) is better suited to smaller objects, and larger (10) for larger objects
            x_start = max(x - margin, 0)
            y_start = max(y - margin, 0)
            x_end = min(x + w + margin, image.shape[1])
            y_end = min(y + h + margin, image.shape[0])

            letter_img = image[y_start:y_end, x_start:x_end]
            filename = f'{image_name}_line_{line_idx + 1}_pos_{pos_idx + 1}.png'       ### CHANGE HERE - can change to another file type other than .png
            save_path = os.path.join(output_dir, filename)

            save_with_dpi(letter_img, save_path, input_dpi)
    
    #create a labelled image showing all the bounding boxes on the input image(s) with line numbers
    image_with_labels = image.copy()
    for line_idx, line in enumerate(lines):
        y = line[0][1]
        label_position = (10, max(y - 10, 0))
        cv.putText(image_with_labels, f"Row {line_idx + 1}", label_position,
                   cv.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)

        for (x, y, w, h) in line:
            cv.rectangle(image_with_labels, (x, y), (x + w, y + h), (0, 255, 0), 2)

    labeled_image_path = os.path.join(output_dir, f'labelled_image_{image_name}.png')   ### CHANGE HERE - can change to another file type other than .png
    
    cv.imwrite(labeled_image_path, image_with_labels)


def process_folder(input_folder, output_folder, default_dpi=DEFAULT_DPI):
    os.makedirs(output_folder, exist_ok=True)
    supported_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', '.pdf')

    images = [f for f in os.listdir(input_folder) if f.lower().endswith(supported_extensions)]
    total_images = len(images)

    if total_images == 0:
        print("No valid images found in folder you specified. Please check the input path and file formats are as you expect!")
        return

    for filename in tqdm(images, desc="Processing Images", unit="image"):
        image_path = os.path.join(input_folder, filename)
        process_image(image_path, output_folder, default_dpi)


#extract the individual objects from the input images and save as individual images
process_folder(input_folder, output_folder)

In [ ]:
###     3. Optical Characters Recognition (OCR) sorting of individual character images into discrete folders
###         (skip if your dataset is not text-based)

#You will need to install tesseract manually first, which is then accessed by Python using the pytesseract package
#Installation documentation can be found here: https://tesseract-ocr.github.io/tessdoc/Installation.html

#load packages
import os
import shutil
import pytesseract
from PIL import Image, ImageFilter, ImageOps
import string
from pathlib import Path

#specify input and output folder locations
try:
    #for .py scripts, where the folder with the image data is also where the script is saved
    root_dir = Path(__file__).resolve().parent
except NameError:
    #for in Jupyter notebooks, where this calls on the current working directory (the folder from which this session was opened)
    root_dir = Path.cwd()

input_folder = root_dir / "extracted"
output_folder = root_dir / "extracted_OCR" #this folder is created as part of this script, you do not have to manually create it
#create an "uncategorised" folder within the output, for anything that cannot be sorted using the OCR script
uncategorised_folder = os.path.join(output_folder, 'uncategorised') #this folder is created as part of this script, you do not have to manually create it

#you could also specify your folders as follows, if your code is not saved in the same location as your image data
#input_folder = r"C:\your\file\path\extracted"                                    ### CHANGE HERE
#output_folder = r"C:\your\file\path\extracted_OCR"                               ### CHANGE HERE
#uncategorised_folder = r"C:\your\file\path\extracted_OCR\uncategorised"          ### CHANGE HERE

#if any of the above folders don't yet exist, the below two lines will create them - or will do nothing if the folders already exist
os.makedirs(output_folder, exist_ok=True)
os.makedirs(uncategorised_folder, exist_ok=True)

#create functions
def preprocess(image):
    image = image.convert('L')  #grayscale
    image = ImageOps.invert(image)  #invert colors to help OCR
    image = image.point(lambda x: 0 if x < 128 else 255, '1')  #binarize
    image = image.filter(ImageFilter.MedianFilter())  #reduce noise
    return image

#expanded list of desired characters for comparison during OCR: letters + numbers + desired punctuation 
whitelist = string.ascii_letters + string.digits + ".,'!?)-*&:;"                        ###CHANGE HERE - add other punctuation characters as required

#make sure that if any punctuation is recognised, that the script will create folders with valid names
#this can be added to, in order to include other punctuation characters
def sanitize_folder_name(char):
    replacements = {
        '/': 'slash',
        '\\': 'backslash', #note: has to be double backslash here to be recognised as a single backslash character and not part of the code
        ':': 'colon',
        '*': 'asterisk',
        '?': 'question',
        '"': 'quote',
        '<': 'less_than',
        '>': 'greater_than',
        '|': 'pipe',
        '.': 'dot',
        ',': 'comma',
        ';': 'semicolon',
        "'": 'apostrophe',
        ')': 'parenthesis',
        ']': 'bracket',
        '-': 'dash',
        '&': 'ampersand',
        '!': 'exclamation'
    }
    return replacements.get(char, char)

#set the beginning number for the tracking statistics
total_images = 0
successful = 0
failed = 0

#run the OCR sorting on the input folder
for filename in os.listdir(input_folder):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):                           ###CHANGE HERE if you have other image file types
        total_images += 1 #add a count to the total number of images being processed tracking statistic
        image_path = os.path.join(input_folder, filename)
        image = Image.open(image_path)
        processed = preprocess(image)

        result = pytesseract.image_to_string(
            processed,
            config=f'--psm 10 -c tessedit_char_whitelist={whitelist}'
        ).strip()

        if result and len(result) == 1: #if there is only a single character recognised in the image, proceed to sort it
            char = result[0]
            successful += 1 #adds a count to the "successful" tracking statistics - only means that an image has been sorted, not necessarily that this is correct!

            if char.isalpha():
                folder_type = 'letters'
                safe_char = sanitize_folder_name(char.lower()) #combine uppercase and lowercase into same folder using lowercase name
            elif char.isdigit():
                folder_type = 'numbers'
                safe_char = sanitize_folder_name(char)
            else:
                folder_type = 'punctuation'
                safe_char = sanitize_folder_name(char)

            char_folder = os.path.join(output_folder, folder_type, safe_char)
            os.makedirs(char_folder, exist_ok=True) #this creates the above folders as necessary, or does nothing if they already exist

            shutil.copy(image_path, os.path.join(char_folder, filename))
        else: #if no characters or multiple characters are recognised in the image, move to the "uncategorised" folder
            failed += 1 #adds a count to the "failed" tracking statistics - means that the image could not be sorted - often due to noise in the image
            shutil.copy(image_path, os.path.join(uncategorised_folder, filename))

#print a sorting report
print(f"Total images: {total_images}")
print(f"Successfully sorted: {successful}")
print(f"Failed to sort:     {failed}")
print("Remember to check the 'uncategorised' folder for all unsorted images!")
print('Also remember that anything "successfully" sorted just means it has been assigned into a folder, not necessarily that this is correct! Make sure to double check every folder!')

In [ ]:
###     4. Manual sorting using Graphic User Interface (GUI)
###         (skip if no further sorting required, or your dataset is not text-based)

#load packages
import os
import cv2 as cv
import shutil
import csv #note this is used only if a .csv log output is desired
from datetime import datetime #note this is used only if a .csv log output is desired
from pathlib import Path

#specify input and output folder locations
try:
    #for .py scripts, where the folder with the image data is also where the script is saved
    root_dir = Path(__file__).resolve().parent
except NameError:
    #for in Jupyter notebooks, where this calls on the current working directory (the folder from which this session was opened)
    root_dir = Path.cwd()

#input folder is the uncategorised folder found within the OCR folder
output_folder = 'extracted_OCR'
uncategorised_folder = os.path.join(output_folder, 'uncategorised')

#create additional folders that are useful to move images into manually during this process
ligature_folder = os.path.join(output_folder, 'ligatures') #remove if you don't have ligatures in your dataset, or change to a different folder name for other character types
incomplete_folder = os.path.join(output_folder, 'incomplete') #folder for any characters that have been extracted in an incomplete state (e.g., in letterpress printing, breaks in the ink impression of a character will mean the outline of a single character is broken into multiple contours)
#if any of these folders don't yet exist, these lines create them automatically, or do nothing if they already exist
os.makedirs(ligature_folder, exist_ok=True)
os.makedirs(incomplete_folder, exist_ok=True)

#if it is useful, you can save a log of the manual classification to a .csv file, which will show a record of which images were classified into what folders
#these lines create the information input into the .csv file, so uncomment this section to create that functionality
#timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
#log_file_path = os.path.join(output_folder, f'manual_classification_log_{timestamp}.csv')
#with open(log_file_path, 'w', newline='') as csvfile:
#    writer = csv.writer(csvfile)
#    writer.writerow(['Filename', 'Character', 'Method'])

#create functions
#make safe folder names for any punctuation characters - not all punctuation may have been recognised in the previous automated OCR sorting stage
#this can be added to, in order to include other punctuation characters
def sanitize_folder_name(char):
    replacements = {
        '/': 'slash', '\\': 'backslash', ':': 'colon', '*': 'asterisk', '?': 'question',
        '"': 'quote', '<': 'lt', '>': 'gt', '|': 'pipe', '.': 'dot', ',': 'comma',
        ';': 'semicolon', "'": 'apostrophe', ')': 'paren', '(': 'paren', ']': 'bracket', '-': 'dash',
        '&': 'ampersand', '!': 'excl', '#':'hash'
    }
    return replacements.get(char, char)

def classify_image(image_path, current, total):
    img = cv.imread(image_path)
    if img is None:
        print(f"Could not open {image_path}")
        return False

    #reminder of the manual classification keyboard options
    #NB: using the shift key with a character will allow you to identify the punctuation characters on a standard keyboard    
    print(f"\nClassifying image {current} of {total}: {os.path.basename(image_path)}")
    print(" - Press the character key to classify.")
    print(" - Spacebar: skip this image")
    print(" - Press [Esc] to quit.")

    #pop up a separate window showing the image to be classified
    #a keyboard press will identify the image and this window will be closed and the next image will open, until the whole folder as been processed (or the sorting is stopped)
    cv.imshow('Classify Image', img)
    key = cv.waitKey(0)
    cv.destroyAllWindows()

    #specify the keyboard buttons for the manual classification
    #NB: using the shift key with a character will allow you to identify the punctuation characters on a standard keyboard
    #it may be possible to add functionality of other keys to move images into other folders (e.g., 'ligatures' or 'incomplete') either by using an otherwise unused key or a combination of keys (e.g., shift + x)
    if key == 27:  #esc key
        return 'quit' #this ends the manual sorting process, so any images that have not yet been sorted will remain in the 'uncategorised' folder
                    #note that the sorting process may also be ended by clicking the 'x' on the pop-up image window
    elif key == 32:  #spacebar
        return None #this skips the image and leaves it in the 'uncategorised' folder
    elif 32 <= key <= 126:  #standard ASCII characters - alphanumeric and common punctuation
        return chr(key)
    else:
        print(f"Warning! Invalid key {key} pressed. Skipping this image.") #in case an invalid key is pressed, the image will simply be skipped and left in the 'uncategorised' folder
        return None
    
images = [f for f in os.listdir(uncategorised_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]         ###CHANGE HERE if you have other image file types

total_images = len(images)
print(f"Starting manual review of {total_images} images.")


for idx, filename in enumerate(images, start=1):
    image_path = os.path.join(uncategorised_folder, filename)
    result = classify_image(image_path, idx, total_images)

    if result == 'quit':
        print("Leaving manual review.")
        break
    elif result is None:
        continue  
    else:
        char = result.lower()  #characters will be added into folders that are named by the lowercase characters - this sorting is case-insensitive
        if len(char) != 1 or not char.isprintable():
            print("Warning! Invalid character entered. Skipping this image.")
            continue

        safe_char = sanitize_folder_name(char)
        if char.isalpha():
            folder_type = 'letters'
        elif char.isdigit():
            folder_type = 'numbers'
        else:
            folder_type = 'punctuation'

        target_folder = os.path.join(output_folder, folder_type, safe_char)
        os.makedirs(target_folder, exist_ok=True) #create folders as needed, or do nothing if they already exist
        shutil.move(image_path, os.path.join(target_folder, filename)) #move the images per the manual classification
        char = result.lower()
        if len(char) != 1 or not char.isprintable():
            print("Warning! Invalid character entered. Skipping this image.")
            continue

        #if it is useful, you can save a log of the manual classification to a .csv file, which will show a record of which images were classified into what folders
        #with open(log_file_path, 'a', newline='') as csvfile:
        #    writer = csv.writer(csvfile)
        #    writer.writerow([filename, char, 'manual'])

print("Manual review complete - all images have been processed.")
#print(f"Manual classification log saved at: {log_file_path}")

#Note that you can re-run this script as many times as needed, and can stop at any point and restart as needed
#unless an 'incomplete' folder has been added to move incomplete character images, likely the 'uncategorised' folder will contain many skipped images, and it is recommended to manually check through this folder in case anything has been inadvertently skipped

#Note that if you have opencv-python-headless installed, the pop-up window will not work properly as this version of the package does not support GUI functionality
#to check which version of opencv you have installed, run the folllowing line
#pip list | findstr opencv
#if you see opencv-python-headless in the list, you will have to uninstall and reinstall using the following lines
#%pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python
#pip install opencv-python
#pip list | findstr opencv
#the list this time should not include opencv-python-headless
#important: now you have reinstalled opencv-python, restart your Python kernel (or close and reopen this code) so that the reinstallation takes place

In [ ]:
#At this point, you may need to manually sort letters into folders so that each letter type is in it's own folder
#For instance letters in the "a" folder may need to be sorted into:
##roman lowercase a
##italic lowercase a
##roman uppercase A
##italic uppercase A
#It is important that by contour extraction, only comparable shapes are included in the same folder, so that the contour coordinates are output correctly
#(you can only compare "RLC a" to "RLC a", you cannot compare "RLC a" to "RUPC A", "ILC a", or "IUPC A")

In [ ]:
###     5. Binarise the individual character images and fill internal holes within the contours

#load packages
import os
import cv2
import numpy as np
from pathlib import Path

#specify the folder to process - this can be a single folder (e.g., "a", "1", "&") or a master folder containing many subfolders (e.g., "letters", "numbers", "punctuation", etc.)
try:
    #for .py scripts, where the folder with the image data is also where the script is saved
    root_dir = Path(__file__).resolve().parent
except NameError:
    #for in Jupyter notebooks, where this calls on the current working directory (the folder from which this session was opened)
    root_dir = Path.cwd()

root_folder = root_dir / "extracted_OCR/letters"

#you could also specify your folders as follows, if your code is not saved in the same location as your image data
#root_folder = r"C:\your\file\path\extracted_OCR\letters"                                    ### CHANGE HERE

#specify whether or not to include all subfolders, or just the root folder itself (if there are no subfolders, either True or False will work)
use_subfolders = True                                                                   ###CHANGE HERE

#specify the image file types
image_exts = ('.png', '.jpg', '.jpeg', '.tif', '.tiff')                                 ###CHANGE HERE if you have other image file types

#create functions
def process_single_folder(input_folder):    
    parent_label = os.path.basename(os.path.normpath(input_folder))
    
    output_folder = os.path.join(
        os.path.dirname(input_folder),
        f"{parent_label} preprocessed" #create an output folder with the same name as the input folder with "preprocessed" added to the end
    )
    os.makedirs(output_folder, exist_ok=True)

    #sense check the locations of the input and output folders
    print(f"Processing: {input_folder}")
    print(f"Saving to:  {output_folder}\n")

    for fname in os.listdir(input_folder):
        if not fname.lower().endswith(image_exts):
            continue

        path = os.path.join(input_folder, fname)
        img_gray = cv2.imread(path, cv2.IMREAD_GRAYSCALE) #read the files in as greyscale
        if img_gray is None:
            continue
        
        #binarise the image using the threshold pixel value 128, halfway between 0 (black) and 255 (white)
        _, binary = cv2.threshold(img_gray, 128, 255, cv2.THRESH_BINARY_INV) #this threshold value can be adjusted if necessary                                                                         
                                                                         #additionally, cv2.THRESH_BINARY_INV can be changed to cv2.THRESH_BINARY to reverse the black and white values
                                                                         #the goal is to create images where the object is in white and the background is black
        #find the contours in the image
        contours, _ = cv2.findContours(binary, cv2.RETR_TREE, cv2.CHAIN_APPROX_NONE)
        #sort the contours from largest to smallest based on the contour area
        sorted_contours = sorted(contours, key=cv2.contourArea, reverse=True)

        #fill the contours in order of size
        filled = np.zeros_like(binary)

        #fill the largest contour (the outline of the character) with white (as the background is black)
        if len(sorted_contours) > 0:
            cv2.drawContours(filled, [sorted_contours[0]], -1, 255, cv2.FILLED)

        #fill any smaller contours based on the proportion of the contour length compared to the largest contour
        if len(sorted_contours) > 1:
            len0 = cv2.arcLength(sorted_contours[0], True)
            len1 = cv2.arcLength(sorted_contours[1], True)
            if len1 / len0 > 0.1:                                                       ###CHANGE HERE - adjust the threshold depending on the size of the holes in the characters, and the characters themselves; every character will require a different threshold
                cv2.drawContours(filled, [sorted_contours[1]], -1, 0, cv2.FILLED) #filled with black, as the second largest contour may be a feature of the characters (e.g., the bowl of the "o")
                                                                                        ###CHANGE HERE - if the character should not have an internal hole (e.g., "c"), then change 0 to 255 to fill with white
            if len(sorted_contours) > 2:
                len2 = cv2.arcLength(sorted_contours[2], True)
                if len2 / len0 > 0.1:                                                   ###CHANGE HERE - adjust the threshold depending on the size of the holes in the characters, and the characters themselves    
                    cv2.drawContours(filled, [sorted_contours[2]], -1, 255, cv2.FILLED) #filled with white, as smaller contours are likely holes inside the character
                                                                                        ###CHANGE HERE - if the character should have two internal holes (e.g., "B"), then change 225 to 0 to fill with black
        
        base, ext = os.path.splitext(fname)
        new_fname = f"{base}_{parent_label}{ext}"
        out_path = os.path.join(output_folder, new_fname)

        #save the filled and binarised image
        cv2.imwrite(out_path, filled)

#either process all subfolders
if use_subfolders:
    #loop through all the subfolders in the root folder
    for dirpath, dirnames, filenames in os.walk(root_folder):
        #only process folders that actually contain images
        if any(f.lower().endswith(image_exts) for f in filenames):
            process_single_folder(dirpath)
#or only process the root folder
else:
    process_single_folder(root_folder)

#you may wish to redo this step multiple times with different threshold settings or contour filling settings to find which works best for your data
#you may require different parameters for different data subsets - e.g., for letterforms, an "o" requires a different hole filling threshold to an "a" due to the size of the internal hole, as can different point sizes of letterforms
#Note: if you re-run the script with different parameters it will overwrite the output files in the "preprocessed" folder unless you change the output folder name

In [ ]:
###     6. Transform the images (if required)

#load packages
import os
import cv2
from pathlib import Path

#specify input and output folder locations
try:
    #for .py scripts, where the folder with the image data is also where the script is saved
    root_dir = Path(__file__).resolve().parent
except NameError:
    #for in Jupyter notebooks, where this calls on the current working directory (the folder from which this session was opened)
    root_dir = Path.cwd()

#Note that these folders have been chosen as examples, and assume no manual separation of the images into "RLC", "ILC", "RUPC", "IUPC" folders
#assuming you have sorted into those folders, the input folder might be something like the following: input_folder = root_dir / "extracted_OCR/letters/RLC u preprocessed"
input_folder = root_dir / "extracted_OCR/letters/u preprocessed"
output_folder = root_dir / "extracted_OCR/letters/u preprocessed transformed" #this folder is created as part of this script, you do not have to manually create it

#you could also specify your folders as follows, if your code is not saved in the same location as your image data
#input_folder = r"C:\your\file\path\preprocessed"                                       ### CHANGE HERE
#output_folder = r"C:\your\file\path\preprocessed transformed"                          ### CHANGE HERE

os.makedirs(output_folder, exist_ok=True) #if the output folder doesn't already exist, this line will create it, or will do nothing if it already exists

#define the parameters of the transformation that you want
#NB: you can set one or both transformations to be applied
rotation_angle = 180            #90, 180, 270                                           ###CHANGE HERE
mirror_axis = 'horizontal'        #'horizontal', 'vertical'                             ###CHANGE HERE

#create functions
#rotation
def rotate_image(img, angle):
    if angle == 180:
        return cv2.rotate(img, cv2.ROTATE_180)
    elif angle == 90:
        return cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
    elif angle == 270:
        return cv2.rotate(img, cv2.ROTATE_90_COUNTERCLOCKWISE)
    else:
        return img  # no rotation
#mirroring
def mirror_image(img, axis):
    if axis == 'horizontal':
        return cv2.flip(img, 1) #mirror on the vertical line, left becomes right
    elif axis == 'vertical':
        return cv2.flip(img, 0) #mirror on the horizontal line, top becomes bottom
    else:
        return img  #no mirroring

for fname in os.listdir(input_folder):
    if not fname.lower().endswith(('.png', '.jpg')):                                    ###CHANGE HERE if you have other image file types
        continue

    input_path = os.path.join(input_folder, fname)
    img = cv2.imread(input_path)

    if img is None:
        print(f"Failed to load image: {fname}")
        continue

    #option 1
    #apply individual transformation
    #NB: uncomment the appropriate line for the transformation that you want to apply
    img_transformed = rotate_image(img, rotation_angle)                                 ###CHANGE HERE
    #img_transformed = mirror_image(img, mirror_axis)                                   ###CHANGE HERE

    #option 2
    #apply multiple transformations
    #NB: if applying both tranformations, comment out the above option 1 lines, and uncomment the below option 2 lines
    #the order of the tranformations can be changed by reordering the lines below - make sure that the first transformation is applied to the original image 'img' and then the second is applied to the transofmred image 'img_transformed'
    #img_transformed = rotate_image(img, rotation_angle)                                ###CHANGE HERE
    #img_transformed = mirror_image(img_transformed, mirror_axis)                       ###CHANGE HERE

    #save to output folder
    output_path = os.path.join(output_folder, fname)
    cv2.imwrite(output_path, img_transformed)

    #print the names of the transformed images as they are saved
    print(f"Saved: {fname}")

In [ ]:
###     7. Extract the contours of the objects in the images and save the coordinates as a .csv file

#Note that this code is currently safe for the extraction of letter shapes with single outlines
#characters with nested outlines (e.g., "B", "o") will need to be treated differently if the internal outlines are intended to be included in morphometric analyses 
#(but this code if safe if only the external outline is of interest)

#Using the visual checks that are part of this script (images output under this cell), you may need to re-run this script multiple times, manually removing specimens from your folder if a full outline is not being recovered
#use the filenames on the images to locate the images with incomplete contours returned, and simply move them into a different folder
#e.g., if inside folder "RLC m preprocessed", create a folder "RLC m preprocessed/incomplete" and move any images into that manually
#then re-run this script, check the plotted contours below again, and if all outlines look complete, output to the .csv file
#note that you can leave save_to_csv = True during this process as it will overwrite each time you re-run the script, or optionally keep as save_to_csv = False until visual checks are complete

#load packages
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

#specify input and output folder locations
try:
    #for .py scripts, where the folder with the image data is also where the script is saved
    root_dir = Path(__file__).resolve().parent
except NameError:
    #for in Jupyter notebooks, where this calls on the current working directory (the folder from which this session was opened)
    root_dir = Path.cwd()

#you can run this code on one or multiple folders, but if multiple, it MUST be the same letter type (e.g., lowercase italic "m")
#the multiple folders therefore allow you to extract contours from multiple different sources at the same time (e.g., from different texts)

#folders if your data is in one place
folders_to_process = [
    root_dir / "extracted_OCR/letters/m preprocessed",
    # root_dir / "folder",
    # root_dir / "folder",
]
#you can also specify the location(s) manually, in case your data is in different locations
#make sure to keep the r before the filepath, which means that the filepath will be read correctly!
#folders_to_process = [
#    r"C:\your\file\path",                                                              ###CHANGE HERE
#    r"C:\your\file\path",
#    r"C:\your\file\path",
#    ]

#.csv file name for the coordinate data output
output_csv = root_dir / "coords.csv" #you can change the .csv file name (e.g., "Baskerville Holy Bible RLC m coords.csv")
#or by specifying a location other than in the root directory
#output_csv = (
#    #r"C:\your\file\path\coords.csv"                                                    ###CHANGE HERE - make sure to keep the r before the filepath, which means that the filepath will be read correctly!
#)

#select whether to save the coordinates to the .csv or not
#it might be advisable to keep this as False the first time you run the script so that you can check the contours visually using the visual check options below
#these will output plots of the contours below this code, allowing you to check that they are outputting correctly, before saving the coordinates to a .csv file
save_to_csv = True

#set the number of coordinates that you want to extract into the .csv from each contour
#this is used here as the number of pseudolandmarks desired, but if a larger number is chosen here, this could be resampled to a smaller number of pseudolandmarks later in R
num_coordinates = 500                                                                  ###CHANGE HERE
#set the threshold pixel value for the binary image used to identify the contours, which is used in the find_largest_contour function, and may need to be adjusted depending on the input images
threshold_value = 128                                                                  ###CHANGE HERE

#create plots to visually check the contours against the binarised images
#note: the speed of the script will be slower with this turned on, so if you just want to save the coordinates, you may wish to turn the visual checks off
#this is really useful to manually weed out any images from the folder where the contour has not been correctly identified
#depending on the dataset, it might be useful to try to improve contour identification rather than remove the images from the folder
        #try: adjusting the threshold value or if necessary, manually edit the images by connecting pixels to improve contour identification
#you may wish to redo Step 5 (binarisation and hole filling) if multiple images have poor contour identification

#1. plot the outlines in green on top of the input binarised images
plot_outlines_on_input_images = True                                                  ###CHANGE HERE

#2. create plots to visually check the contours only
plot_processed_outlines = True                                                        ###CHANGE HERE

#3. plot the outlines stacked on top of each other
#note that the script doesn't apply any form of Procrustes alignment, so the contours will be plotted in their original positions rather than being positioned relative to one another
plot_stacked_outlines = True                                                          ###CHANGE HERE
#select whether or not to save the stacked outline as an .svg file
save_stacked_outlines = True                                                                ###CHANGE HERE
stacked_outlines_image_path = os.path.splitext(output_csv)[0] + "_stacked_contours.svg"     ###CHANGE HERE make sure to change the file name to your specific dataset

#direction is enforced after converting from OpenCV image coordinates to standard Cartesian-style coordinates
# "ccw" = counter-clockwise
# "cw"  = clockwise
# None  = keep OpenCV's original winding order after y-axis inversion
contour_direction = "ccw"

#starting point options:
# "rightmost"          # highest x value
# "leftmost"           # lowest x value
# "topmost"            # highest y value after y-axis inversion
# "bottommost"         # lowest y value after y-axis inversion
# "closest_to_origin"  # closest point to (0, 0)
#
#you can also use an angle in degrees, e.g.
# start_point = 0      # right side of the contour, relative to centroid
# start_point = 90     # top of the contour, relative to centroid
# start_point = 180    # left side of the contour, relative to centroid
# start_point = -90    # bottom of the contour, relative to centroid
start_point = "rightmost"

#create functions
#find the largest contour in the binary image which should be the outline of the object
def find_largest_contour(binary_img):
    
    contours, _ = cv2.findContours(
        binary_img,
        cv2.RETR_TREE, #find all contours and their internal hierarchies
        cv2.CHAIN_APPROX_NONE #store all contour points in the path
    )

    if not contours:
        return None

    largest = max(contours, key=lambda c: cv2.arcLength(c, True))

    # Convert from OpenCV's shape, N x 1 x 2, to a simpler N x 2 array.
    return largest[:, 0, :]

#function to resample the contour using the specified number of points
def resample_contour(contour, num_points):
    if contour is None or len(contour) == 0:
        return np.zeros((num_points, 2))

    if len(contour) < 2:
        return np.tile(contour[0], (num_points, 1))

    distances = np.sqrt(np.sum(np.diff(contour, axis=0) ** 2, axis=1))
    cum_dist = np.insert(np.cumsum(distances), 0, 0)
    total_length = cum_dist[-1]

    if total_length == 0:
        return np.tile(contour[0], (num_points, 1))

    even_dist = np.linspace(0, total_length, num_points)

    x_interp = np.interp(even_dist, cum_dist, contour[:, 0])
    y_interp = np.interp(even_dist, cum_dist, contour[:, 1])

    return np.stack((x_interp, y_interp), axis=1)

#determine the direction of the contour, where a positive value = counter-clockwise and a negative value = clockwise
def signed_contour_area(contour):
    x = contour[:, 0]
    y = contour[:, 1]

    return 0.5 * np.sum(
        x * np.roll(y, -1) -
        np.roll(x, -1) * y
    )

#enforce the contour direction
#it doesn't matter which direction is chosen as long as all contours have a consistent direction
def enforce_contour_direction(contour, direction="ccw"):
    if direction is None:
        return contour

    direction = direction.lower()

    if direction not in {"ccw", "counterclockwise", "counter-clockwise", "cw", "clockwise"}:
        raise ValueError(
            "contour_direction must be 'ccw', 'cw', or None."
        )

    area = signed_contour_area(contour)

    if area == 0:
        return contour

    wants_ccw = direction in {"ccw", "counterclockwise", "counter-clockwise"}
    is_ccw = area > 0

    if wants_ccw != is_ccw:
        contour = contour[::-1].copy()

    return contour


def get_start_index(contour, start_point="rightmost"):                                 ####CHANGE HERE to adjust the desired start point of the contour
    if isinstance(start_point, (int, float)):
        centre = np.mean(contour, axis=0)
        centred = contour - centre

        target_angle = np.deg2rad(start_point)
        contour_angles = np.arctan2(centred[:, 1], centred[:, 0])

        angular_difference = np.abs(
            np.angle(np.exp(1j * (contour_angles - target_angle)))
        )

        return int(np.argmin(angular_difference))

    start_point = start_point.lower()

    if start_point == "rightmost":
        return int(np.argmax(contour[:, 0]))

    if start_point == "leftmost":
        return int(np.argmin(contour[:, 0]))

    if start_point == "topmost":
        return int(np.argmax(contour[:, 1]))

    if start_point == "bottommost":
        return int(np.argmin(contour[:, 1]))

    if start_point == "closest_to_origin":
        distances = np.linalg.norm(contour, axis=1)
        return int(np.argmin(distances))

    raise ValueError(
        "start_point must be 'rightmost', 'leftmost', 'topmost', "
        "'bottommost', 'closest_to_origin', or a numeric angle in degrees."
    )

#set a specific contour start point by rolling the contour array if the desired start point is not already the first point in the contour
def set_contour_start(contour, start_point="rightmost"):
    start_idx = get_start_index(contour, start_point=start_point)
    return np.roll(contour, -start_idx, axis=0)

#plot the identified contour in green overlain on an image of the binarised input
def display_contours_on_input_images(image, contours, title=""):
    
    img_color = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)

    for i, contour in enumerate(contours): 
        if contour is not None:
            cv2.drawContours(
                img_color,
                [contour.reshape(-1, 1, 2).astype(np.int32)],
                -1,
                (0, 255, 0) if i == 0 else (0, 0, 255),
                1
            )

    plt.imshow(img_color)
    plt.title(f"{title} (Contour=Green)")
    plt.axis("off")
    plt.show()

#plot all the contours as single lines stacked on top of each other
def display_stacked_contours(
    contour_list,
    title="Stacked outlines",
    colour="black", #can change the colour of the contours
    alpha=1.0, #alpha controls the opacity/transparency of the plotted contours, with 1.0 being fully opaque and lower values being more transparent
    save_svg_path=None,
    show=True
):
   
    if not contour_list:
        print("No contours available to plot.")
        return

    fig, ax = plt.subplots(figsize=(7, 7))

    for contour in contour_list:
        ax.plot(
            contour[:, 0],
            contour[:, 1],
            color=colour,
            alpha=alpha,
            linewidth=1,
            antialiased=False
        )

    ax.set_aspect("equal", adjustable="box")
    ax.axis("off")

    if title is not None:
        ax.set_title(title)

    plt.tight_layout()

    if save_svg_path is not None:
        fig.savefig(
            save_svg_path,
            format="svg",                                                              ###CHANGE HERE if you want to save in a different format, e.g., "png"
            bbox_inches="tight", #keep the space around the image minimal
            pad_inches=0, #add padding around the image if necessary
            transparent=True #make the background transparent
        )
        print(f"Saved stacked .svg to: {save_svg_path}")

    plt.show()

#create objects
#all_rows will store the rows of coordinate data
all_rows = []
#store the unflattened contour data as an object so that it can be plotted
unflattened_contours = []
#set the coordinate columns to number sequentially from x_001, y_001, x_002, y_002, etc.
coord_columns = [
    f"{axis}_{i + 1:03d}"
    for i in range(num_coordinates)
    for axis in ("x", "y")
]

#process the folder(s)
for folder in folders_to_process:

    #if the folder can't be found, it will not process - this is likely caused by a typo in the folder path
    if not os.path.isdir(folder):
        print(f"Skipping missing folder: {folder}, please check the folder path for typos.")
        continue

    for fname in sorted(os.listdir(folder)):   

        img_path = os.path.join(folder, fname)

        #read the image in as a greyscale image
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        #skip over non-image files
        if img is None:
            print(f"Skipping unreadable file: {fname}, please check image properties and input filepath for typos.")
            continue

        #threshold the image to create a binarised version to look for contours in
        #although binarisation has already occured in a previous stage, this ensures safety of the script in case something has been missed or did not work fully the first time
        _, binary = cv2.threshold( #first value returned is the threshold value used (so we can ignore using "_"), second is the binary image
            img,
            threshold_value, #this is the threshold value set at the start of the script
            255,
            cv2.THRESH_BINARY
        )

        #find the largest contour in the image, which should be the outline of the object in the image
        contour = find_largest_contour(binary)

        if contour is None or len(contour) < 2:
            print(f"No valid contour found in: {fname}")
            continue

        #plot the contour on top of the binarised image to check that the contour has been correctly identified
        if plot_outlines_on_input_images:
            display_contours_on_input_images(
                binary,
                [contour],
                title=f"Original contour: {fname}"
            )

        #make sure that the contour data is in the right format for subsequent processing
        contour = contour.astype(np.float64, copy=True)

        #convert from OpenCV image coordinate space to standard Cartesian-style plotting coordinate space (y-axis inversion)
        contour[:, 1] = -contour[:, 1]

        #enforce the direction of the contours
        contour = enforce_contour_direction(
            contour,
            direction=contour_direction
        )

        #enforce the start point of the contours
        contour = set_contour_start(
            contour,
            start_point=start_point
        )

        #resample the number of points in the contour to the specified number of coordinates
        contour = resample_contour(
            contour,
            num_points=num_coordinates
        )

        #store the unflattened contour data so that this can be plotted within this code
        unflattened_contours.append(contour)

        #plotting the contours stacked on each other
        #this has to be done before the contours are flattened for saving into the .csv coordinate data file
        if plot_processed_outlines:
            display_stacked_contours(
                [contour],
                title=f"Processed outline: {fname}",
                colour="black", #can change colour of the plotted contour
                alpha=1.0 #this changes the opacity/transparency of the contour, 1.0 is fully opaque, lower is more transparent
            )

        #flatten the coordinate data shape so that it can be saved into the .csv file with each row being one line of coordinate data per specimen
        flat_coords = contour.flatten()


        #add the metadata, which here is only the filename, into the row of coordinate data that will be saved to the .csv file along with the coordinate data
        row = {
            "filename": fname
        }

        #add in the coordinate data into each row
        row.update(
            {
                col: value
                for col, value in zip(coord_columns, flat_coords)
            }
        )

        #add every row in turn to the object all_rows
        all_rows.append(row)

#create a dataframe of all the coordinate rows, which have been saved in the object all_rows
df = pd.DataFrame(all_rows)

#parameters for saving the coordinate data in a .csv file
if save_to_csv:
    if len(df) > 0:
        df.to_csv(output_csv, index=False)
        print(f"Saved {len(df)} outlines to: {output_csv}")
    else:
        print("No outlines were processed, and no .csv was written. Please check input filepaths for typos.")

#visualising the contours
if plot_stacked_outlines or save_stacked_outlines:
    display_stacked_contours(
        unflattened_contours,
        title=f"Stacked outlines {folder}",
        colour="black",
        alpha=1.0,
        save_svg_path=stacked_outlines_image_path if save_stacked_outlines else None,
        show=plot_stacked_outlines
    )

#print the number of outlines that have been processed and saved to the .csv
print(f"\nTotal outlines processed: {len(df)}")